In [ ]:
# Databricks notebook source

# ================================================================
# CELL 1
# PURPOSE:
#   Initialize pipeline monitoring and determine the latest
#   available pipeline run date from the Gold run-details view.
#
# IMPORTANT:
#   We intentionally DO NOT use CURRENT_DATE().
#
#   The monitoring date is derived directly from the pipeline
#   execution data. This avoids timezone/date-boundary problems.
# ================================================================


RUN_DETAILS_VIEW = (
    "prdrzranalytics.lab42."
    "sdi_vw_pipelineMonitoring_gold_bqUcRunDetails_daily"
)

HEALTH_SUMMARY_VIEW = (
    "prdrzranalytics.lab42."
    "sdi_vw_pipelineMonitoring_gold_bqUcHealthSummary_daily"
)

FAILED_TABLES_VIEW = (
    "prdrzranalytics.lab42."
    "sdi_vw_pipelineMonitoring_gold_bqUcFailedTables_daily"
)


print("=" * 80)
print("SDI BQ -> UC PIPELINE MONITORING")
print("=" * 80)

print("Run details view:")
print(RUN_DETAILS_VIEW)

print("\nHealth summary view:")
print(HEALTH_SUMMARY_VIEW)

print("\nFailed tables view:")
print(FAILED_TABLES_VIEW)

print("\nDetermining latest available pipeline run date...")


latest_run_df = spark.sql(
    f"""
    SELECT

        MAX(run_date) AS latest_run_date,

        MAX(job_run_ts) AS latest_job_run_ts

    FROM {RUN_DETAILS_VIEW}
    """
)


latest_run_row = latest_run_df.first()


if latest_run_row is None or latest_run_row["latest_run_date"] is None:

    raise Exception(
        "PIPELINE MONITORING ERROR: "
        "No pipeline execution records were found in the monitoring view."
    )


latest_run_date = latest_run_row["latest_run_date"]

latest_job_run_ts = latest_run_row["latest_job_run_ts"]


print("\nLatest pipeline date found:")
print(latest_run_date)

print("\nLatest individual execution timestamp:")
print(latest_job_run_ts)

print("\nMonitoring will evaluate ALL table executions for:")
print(latest_run_date)

print("=" * 80)

In [ ]:
# ================================================================
# CELL 2
# PURPOSE:
#   Retrieve and display the overall pipeline health summary for
#   the latest available pipeline run date.
# ================================================================


print("=" * 80)
print("PIPELINE HEALTH SUMMARY")
print("=" * 80)

print(f"Monitoring run date: {latest_run_date}")


health_df = spark.sql(
    f"""
    SELECT

        run_date,

        total_execution_attempts,

        total_tables,

        successful_tables,

        failed_tables,

        skipped_tables,

        unknownStatus_tables,

        success_rate_pct,

        rowCountMismatch_tables,

        schemaChange_tables,

        total_bq_rows,

        total_loaded_rows,

        total_rowCountDifference,

        first_jobRun_ts,

        last_jobEnd_ts,

        total_elapsed_minutes

    FROM {HEALTH_SUMMARY_VIEW}

    WHERE run_date = DATE('{latest_run_date}')
    """
)


health_rows = health_df.collect()


if len(health_rows) == 0:

    raise Exception(
        f"PIPELINE MONITORING ERROR: "
        f"No health summary was found for {latest_run_date}."
    )


health = health_rows[0]


print("\nPipeline health metrics:")
print("-" * 80)

print(
    f"Total execution attempts : "
    f"{health['total_execution_attempts']}"
)

print(
    f"Total tables             : "
    f"{health['total_tables']}"
)

print(
    f"Successful tables        : "
    f"{health['successful_tables']}"
)

print(
    f"Failed tables            : "
    f"{health['failed_tables']}"
)

print(
    f"Skipped tables           : "
    f"{health['skipped_tables']}"
)

print(
    f"Unknown status tables    : "
    f"{health['unknownStatus_tables']}"
)

print(
    f"Success rate             : "
    f"{health['success_rate_pct']}%"
)

print(
    f"Row-count mismatches     : "
    f"{health['rowCountMismatch_tables']}"
)

print(
    f"Tables with schema change: "
    f"{health['schemaChange_tables']}"
)

print(
    f"Total BQ rows            : "
    f"{health['total_bq_rows']}"
)

print(
    f"Total UC loaded rows     : "
    f"{health['total_loaded_rows']}"
)

print(
    f"Total row difference     : "
    f"{health['total_rowCountDifference']}"
)

print(
    f"First execution          : "
    f"{health['first_jobRun_ts']}"
)

print(
    f"Last execution           : "
    f"{health['last_jobEnd_ts']}"
)

print(
    f"Total elapsed minutes    : "
    f"{health['total_elapsed_minutes']}"
)


print("-" * 80)


# Display a formatted Databricks table as well.
display(health_df)

In [ ]:
# ================================================================
# CELL 3
# PURPOSE:
#   Retrieve all tables whose latest execution for the monitoring
#   date remains in Failed status.
#
#   Print:
#       - BQ table name
#       - UC target table
#       - batch
#       - error message
#
#   The DataFrame is also displayed for easier troubleshooting.
# ================================================================


print("=" * 80)
print("FAILED TABLE CHECK")
print("=" * 80)

print(f"Checking failed tables for: {latest_run_date}")


failed_df = spark.sql(
    f"""
    SELECT

        run_date,

        job_run_ts,

        job_end_ts,

        batch_number,

        bq_table,

        uc_table,

        run_type,

        status,

        bq_rows,

        loaded_rows,

        row_count_difference,

        elapsed_sec,

        error_message

    FROM {FAILED_TABLES_VIEW}

    WHERE run_date = DATE('{latest_run_date}')

    ORDER BY

        batch_number,

        bq_table
    """
)


failed_rows = failed_df.collect()

failed_table_count = len(failed_rows)


print(
    f"\nNumber of failed tables detected: "
    f"{failed_table_count}"
)


if failed_table_count == 0:

    print("\nSUCCESS")
    print(
        "No BQ -> UC table copy failures were detected "
        "for the latest pipeline date."
    )

else:

    print("\nFAILED TABLES")
    print("-" * 80)


    for index, row in enumerate(failed_rows, start=1):

        print(f"\nFailure #{index}")

        print(
            f"BQ table       : "
            f"{row['bq_table']}"
        )

        print(
            f"UC table       : "
            f"{row['uc_table']}"
        )

        print(
            f"Batch number   : "
            f"{row['batch_number']}"
        )

        print(
            f"Job run time   : "
            f"{row['job_run_ts']}"
        )

        print(
            f"BQ rows        : "
            f"{row['bq_rows']}"
        )

        print(
            f"Loaded rows    : "
            f"{row['loaded_rows']}"
        )

        print(
            f"Error message  : "
            f"{row['error_message']}"
        )

        print("-" * 80)


# Show the failed records as a Databricks result table.
display(failed_df)

In [ ]:
# ================================================================
# CELL 4
# PURPOSE:
#   Final monitoring result.
#
#   If failed_table_count == 0:
#       Notebook completes successfully.
#
#   If failed_table_count > 0:
#       Raise an Exception.
#
#   Raising the exception causes the Databricks Workflow task to
#   move into FAILED state, which triggers the configured
#   task-failure notification.
# ================================================================


print("=" * 80)
print("FINAL PIPELINE MONITORING RESULT")
print("=" * 80)


if failed_table_count == 0:

    print(
        f"PASS: BQ -> UC pipeline monitoring completed successfully "
        f"for {latest_run_date}."
    )

    print("Failed table count: 0")

    print(
        "No failed table copies require investigation."
    )

    print("=" * 80)


else:

    print(
        f"FAIL: {failed_table_count} failed BQ -> UC "
        f"table copy/copies detected."
    )

    print("\nFailed BQ tables:")


    # ------------------------------------------------------------
    # Create list of failed table names.
    # ------------------------------------------------------------

    failed_table_names = [
        row["bq_table"]
        for row in failed_rows
    ]


    for table_name in failed_table_names:

        print(f" - {table_name}")


    # ------------------------------------------------------------
    # Create readable exception text.
    #
    # Keep the message reasonably small because this text can
    # appear in Databricks Job failure notifications.
    # ------------------------------------------------------------

    max_tables_in_message = 20


    alert_table_names = failed_table_names[
        :max_tables_in_message
    ]


    table_list_text = ", ".join(
        alert_table_names
    )


    if failed_table_count > max_tables_in_message:

        remaining_count = (
            failed_table_count
            - max_tables_in_message
        )

        table_list_text += (
            f", and {remaining_count} additional table(s)"
        )


    error_message = (
        f"BQ -> UC PIPELINE ALERT | "
        f"Run Date: {latest_run_date} | "
        f"Failed Tables: {failed_table_count} | "
        f"Tables: {table_list_text}. "
        f"Review "
        f"{FAILED_TABLES_VIEW} "
        f"for error details."
    )


    print("\nAlert message:")
    print(error_message)

    print("=" * 80)


    # ------------------------------------------------------------
    # Intentionally fail the Databricks task.
    # ------------------------------------------------------------

    raise Exception(error_message)

In [ ]:
# ================================================================
# CELL 5
# PURPOSE:
#   This cell is reached only when all monitoring checks pass.
# ================================================================


print("=" * 80)
print("SDI PIPELINE MONITORING COMPLETE")
print("=" * 80)

print(f"Run date monitored : {latest_run_date}")

print("Monitoring status  : SUCCESS")

print("Failed tables      : 0")

print("=" * 80)